In [ ]:
import torch.utils.data as data
import numpy as np
import os
import sys
from PIL import Image
import torch
import random
import math
from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F

from torch.autograd import Variable
from torchinfo import summary

print(torchvision.__version__)
print(torch.__version__)

data_root_dir = "/scratch1/zeyut/eat_detection/"

In [6]:
torch.cuda.current_device()
[torch.cuda.get_device_name(idx) for idx in range(torch.cuda.device_count())]

['Tesla V100-PCIE-16GB']

In [3]:
LABEL_NUM = 3

RAW_FRAME_LOC = data_root_dir + "VideoData_rawFrames/"  
FRAME_LOC = data_root_dir + "VideoData_independent/"  
WIDTH = 224
HEIGHT = 224
CHANNEL = 3
LABEL_NUM = 3
LABEL_TABLE = {"bite": 0, "drink": 1, "non_intake": 2}

seq_len = 16
stride = 8
model_type = 1
batch_size = 16
#train_video_list = ["p207_c3","p176_c1","p179_c3","p177_c2","p176_c2"]
train_video_list = [f for f in os.listdir(FRAME_LOC+"train_set") if f.startswith("p")]
val_video_list = [f for f in os.listdir(FRAME_LOC+"val_set") if f.startswith("p")]

In [4]:
weights = []
weight_type =5
class_counts =  [333634, 136993, 1964285]
class_counts = np.array(class_counts)
a = 10000
total = sum(class_counts)
for i in range(LABEL_NUM):
    if class_counts[i] == 0:
        ret.append(0)
    else:
        if weight_type == 1:    
            weights.append(1/LABEL_NUM)
        elif weight_type == 2:
            weights.append(a/(1+LABEL_NUM*class_counts[i]/total))
        elif weight_type == 3:
            weights.append(a/class_counts[i])
        elif weight_type == 4:
            weights.append(a/class_counts[i]**0.5)
        elif weight_type == 5:
            beta = 0.99999
            weights.append((1-beta)/(1-beta**class_counts[i]))

weights = np.array(weights) 
weights = weights/np.sum(weights)     
print(f"weights: {weights}")

weights: [0.30698827 0.3969419  0.29606983]


In [5]:
class FrameSequenceDataset(data.Dataset):
    def __init__(self,root_path,video_list,seq_len,stride,model_type,transform,test_mode=False):
        'Initialization'
        self.root_path = root_path
        self.model_type = model_type
        self.transform = transform
        self.test_mode = test_mode
        self.sample_list = []
        self.label_list = []
        self._get_data_list(video_list, seq_len, stride, model_type)
        
    def __len__(self):
        return len(self.sample_list)

    def __getitem__(self, idx):
        frame_list, labels = self.sample_list[idx], self.label_list[idx]
        frames = self._get_frames(frame_list)
        frames = torch.stack([transforms.functional.to_tensor(frame) for frame in frames])
        if not self.test_mode and self.transform is not None:
            frames = self.transform(frames)
        return frames, labels

    def _get_frames(self, frame_list):
        frames = []
        for frame_loc in frame_list:
            frames.append(Image.open(frame_loc).convert('RGB'))
        return frames

    def _get_data_list(self, video_list, seq_len, stride, model_type):
        for video in video_list:
            frame_locs = []
            frame_labels = []
            f = open(self.root_path + video + "/gt_frame_3labels.txt","r")
            gt_frame = [str.split(line, "\t") for line in f.readlines()]
            for frame_info in gt_frame:
                frame_locs.append(self.root_path + video + "/" + frame_info[0])
                cur_label_idx = LABEL_TABLE[str.split(frame_info[1], "\n")[0]]
                frame_labels.append(cur_label_idx)
            for i in range(0, len(frame_locs)-seq_len, stride):
                self.sample_list.append(frame_locs[i:i+seq_len])
                if model_type == 1:
                    self.label_list.append(np.array(frame_labels[i:i+seq_len]))
                elif model_type == 2:
                    self.label_list.append(np.array(frame_labels[i+seq_len-1]))
            
        self.label_list = np.array(self.label_list)
        self.sample_list = np.array(self.sample_list)
def denormalize(video_tensor):
    """
    Undoes mean/standard deviation normalization, zero to one scaling,
    and channel rearrangement for a batch of images.
    """
    inverse_normalize = transforms.Normalize(
            mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
            std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
    )
    return (inverse_normalize(video_tensor) * 255.).type(torch.uint8).permute(0, 2, 3, 1).numpy()


In [6]:
preprocess = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5), 
            transforms.ColorJitter(brightness=0.4),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
train_set = FrameSequenceDataset(
        root_path=FRAME_LOC+"train_set/",
        video_list=train_video_list,
        seq_len=seq_len,
        stride=stride*8,
        model_type=1,
        transform=preprocess,
        test_mode=False
        )
train_loader = data.DataLoader(
        dataset=train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

In [19]:
val_set = FrameSequenceDataset(
        root_path=FRAME_LOC+"val_set/",
        video_list=val_video_list,
        seq_len=seq_len,
        stride=stride,
        model_type=1,
        transform=preprocess,
        test_mode=False
        )
val_loader = data.DataLoader(
        dataset=val_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

### Test x3d, cnn_lstm, slowfast

In [1]:
import torch.utils.data as data
import numpy as np
import os
import sys
from PIL import Image
import torch
import random
import math
from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F

from torch.autograd import Variable
from torchinfo import summary

from fvcore.nn import FlopCountAnalysis

print(torchvision.__version__)
print(torch.__version__)

data_root_dir = "/scratch1/zeyut/eat_detection/"

sys.path.append("../reimplementation/models/")

import x3d, cnn_lstm, slowfast

0.9.1+cu102
1.9.1+cu102


In [8]:
input = torch.randn(1, 16, 3, 224, 224,dtype=torch.float).cuda()

In [9]:
input.size()[0]+input.size()[1]

17

In [5]:
slowfast_model = slowfast.generate_model('resnet101', 4, class_num=3).cuda()

In [28]:
flops = FlopCountAnalysis(slowfast_model, input)
print(f"GFLOPs: {flops.total() // 1000000000}")

Unsupported operator aten::max_pool3d encountered 2 time(s)
Unsupported operator aten::add_ encountered 66 time(s)
Unsupported operator aten::adaptive_avg_pool3d encountered 2 time(s)


GFLOPs: 48


In [ ]:
summary(slowfast_model, input_size=(16, 16, 3, 224, 224)) #-1, C, T, H, W

In [14]:
x3d_model = x3d.generate_model('M',n_classes=3, n_input_channels=3, task='loc', dropout=0.5, base_bn_splits=1).cuda()

In [15]:
flops = FlopCountAnalysis(x3d_model, input)
print(f"GFLOPs: {flops.total() // 1000000000}")

/home/zeyut/.conda/envs/torch-1.8/lib/python3.8/site-packages/torch/_tensor.py:575: UserWarning: floor_divide is deprecated, and will be removed in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values.
To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor'). (Triggered internally at  ../aten/src/ATen/native/BinaryOps.cpp:467.)
  return torch.floor_divide(self, other)
Unsupported operator aten::mul encountered 183 time(s)
Unsupported operator aten::add encountered 84 time(s)
Unsupported operator aten::adaptive_avg_pool3d encountered 16 time(s)
Unsupported operator aten::sigmoid encountered 15 time(s)
Unsupported operator prim::PythonOp.SwishEfficient encountered 26 time(s)
Unsupported operator aten::add_ encountered 26 time(s)
The following submodules of the model were never called during the trace

GFLOPs: 5


In [20]:
load_ckpt = torch.load('../reimplementation/pre_trained/x3d_m.pyth') # SET steps=0, OR REMOVE optimizer,lr_sched LOADING


In [7]:
input

<bound method Kernel.raw_input of <ipykernel.ipkernel.IPythonKernel object at 0x151ea1edac40>>

In [33]:
summary(x3d_model, input_size=(1, 16, 3, 224, 224)) #-1, C, T, H, W

Layer (type:depth-idx)                   Output Shape              Param #
├─Conv3d: 1-1                            [1, 32, 16, 112, 112]     864
├─Conv3d: 1-2                            [1, 32, 16, 112, 112]     160
├─SubBatchNorm3d: 1-3                    [1, 32, 16, 112, 112]     --
|    └─BatchNorm3d: 2-1                  [1, 32, 16, 112, 112]     --
├─ReLU: 1-4                              [1, 32, 16, 112, 112]     --
├─Sequential: 1-5                        [1, 32, 16, 56, 56]       --
|    └─Bottleneck: 2-2                   [1, 32, 16, 56, 56]       --
|    |    └─Conv3d: 3-1                  [1, 72, 16, 112, 112]     2,304
|    |    └─BatchNorm3d: 3-2             [1, 72, 16, 112, 112]     144
|    |    └─ReLU: 3-3                    [1, 72, 16, 112, 112]     --
|    |    └─Conv3d: 3-4                  [1, 72, 16, 56, 56]       1,944
|    |    └─BatchNorm3d: 3-5             [1, 72, 16, 56, 56]       144
|    |    └─AdaptiveAvgPool3d: 3-6       [1, 72, 1, 1, 1]          --
|    

In [19]:
cnn_lstm_model = cnn_lstm.CNN_LSTM(basemodel='resnet50').cuda()

In [13]:
summary(cnn_lstm_model, input_size=(1, 16, 3, 224, 224)) #-1, T, C, H, W

Layer (type:depth-idx)                   Output Shape              Param #
├─Spatial_Encoder: 1-1                   [1, 16, 512, 1, 1]        --
|    └─Sequential: 2-1                   [16, 512, 1, 1]           --
|    |    └─Conv2d: 3-1                  [16, 64, 112, 112]        9,408
|    |    └─BatchNorm2d: 3-2             [16, 64, 112, 112]        128
|    |    └─ReLU: 3-3                    [16, 64, 112, 112]        --
|    |    └─MaxPool2d: 3-4               [16, 64, 56, 56]          --
|    |    └─Sequential: 3-5              [16, 64, 56, 56]          221,952
|    |    └─Sequential: 3-6              [16, 128, 28, 28]         1,116,416
|    |    └─Sequential: 3-7              [16, 256, 14, 14]         6,822,400
|    |    └─Sequential: 3-8              [16, 512, 7, 7]           13,114,368
|    |    └─AdaptiveAvgPool2d: 3-9       [16, 512, 1, 1]           --
├─Flatten: 1-2                           [1, 16, 512]              --
├─BatchNorm1d: 1-3                       [1, 16, 512] 

In [20]:
flops = FlopCountAnalysis(cnn_lstm_model, input)
print(f"GFLOPs: {flops.total() // 1000000000}")

Unsupported operator aten::mul encountered 1 time(s)
Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 16 time(s)
Unsupported operator aten::lstm encountered 1 time(s)
Unsupported operator aten::softmax encountered 1 time(s)


GFLOPs: 66


In [21]:
flops.by_operator()

Counter({'conv': 65394180096,
         'batch_norm': 889282560,
         'adaptive_avg_pool2d': 1605632,
         'linear': 6144})

In [83]:
resnet101 = torchvision.models.resnet101(pretrained=True).cuda()
summary(model, input_size=(1, 3, 224, 224))
resnet50 = torchvision.models.resnet50(pretrained=True).cuda()
summary(resnet, input_size=(1, 3, 224, 224))

Downloading: "https://download.pytorch.org/models/resnet101-5d3b4d8f.pth" to /home/zeyut/.cache/torch/hub/checkpoints/resnet101-5d3b4d8f.pth


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [6]:
class TimeDistributed(nn.Module):
    def __init__(self, module, batch_first=True):
        super(TimeDistributed, self).__init__()
        self.module = module
        self.batch_first = batch_first
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
        input = x.view(batch_size * time_steps, C, H, W)
        output = self.module(input)
        output = output.view(batch_size, time_steps, -1)
        if self.batch_first is False:
            output = output.permute(1, 0, 2)
        return output

    
class Spatial_Encoder(nn.Module):
    def __init__(self,basemodel='resnet34'):
        super(Spatial_Encoder, self).__init__()
        self._prepare_basemodel(basemodel)
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
        x = x.view(batch_size * time_steps, C, H, W)
        x = self.net(x)
        new_C, new_H, new_W = x.size()[-3:]
        output = x.view(batch_size, time_steps, new_C, new_H, new_W)
        return output
    def _prepare_basemodel(self,basemodel):
        if basemodel == "resnet34":
            model = torchvision.models.resnet34(pretrained=True)
        if basemodel == "resnet50":
            model = torchvision.models.resnet50(pretrained=True)
        if basemodel == "resnet101":
            model = torchvision.models.resnet101(pretrained=True)
        module_list = list(model.children())
        del module_list[-1]
        self.net = nn.Sequential(*module_list)

In [9]:
LABEL_NUM = 3
class RES_LSTM(nn.Module):
    def __init__(self,seq_len=16,basemodel='resnet34'):
        super(RES_LSTM, self).__init__()
        self.encoder = Spatial_Encoder(basemodel)
        if basemodel=='resnet34':
            encoder_size = 512
        if basemodel=='resnet50':
            encoder_size = 2048
        self.lstm = nn.LSTM(input_size=encoder_size,
                            hidden_size=128,
                            num_layers=2,
                            batch_first=True)
        self.batch_norm = nn.BatchNorm1d(num_features=seq_len)
        self.flatten = nn.Flatten(start_dim=2,end_dim=-1)
        self.fc = nn.Sequential(nn.Linear(128, LABEL_NUM),
                                nn.ReLU())
        self.act = nn.Softmax(dim=-1)
    def forward(self, x):
        x = self.encoder(x)
        x = self.flatten(x)
        x = self.batch_norm(x)
        x,(hn, cn) = self.lstm(x)
        x = self.fc(x)
        output = self.act(x)
        return output

In [13]:
res_lstm = RES_LSTM(seq_len=16).cuda()
flops = FlopCountAnalysis(res_lstm, input)
print(f"GFLOPs: {flops.total() // 1000000000}")


Unsupported operator aten::mul encountered 1 time(s)
Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 16 time(s)
Unsupported operator aten::lstm encountered 1 time(s)
Unsupported operator aten::softmax encountered 1 time(s)


GFLOPs: 58


In [ ]:
summary(res_lstm, input_size=(32,2, CHANNEL, HEIGHT, WIDTH))

In [64]:
def class_weights(label_list, weight_type):
    class_counts = []
    for label in range(LABEL_NUM):
        class_counts.append(np.sum(label_list==label))
    class_counts = np.array(class_counts)
    total = np.sum(class_counts)
    ret = []
    for i in range(LABEL_NUM):
        if class_counts[i] == 0:
            ret.append(0)
        else:
            if weight_type == 1:
                """
                version 1: n/(m*c(i))
                where n: total sample number, m: number of classes. c(i): number of samples belonging to the class
                """
                ret.append(total/(class_counts[i] * LABEL_NUM) / np.sum(total/(class_counts[class_counts!=0] * LABEL_NUM)))
            elif weight_type == 2:
                """
                version 2: 1/(1+c(i)/n)
                """
                ret.append(1/(1+LABEL_NUM*class_counts[i]/total) / np.sum(1/(1+LABEL_NUM*class_counts[class_counts!=0]/total)))
            elif weight_type == 3:
                """
                version 3: 1/c(i) / sum(1/c(i)) (Inverse Number of Sample)
                """
                ret.append(1/class_counts[i] / np.sum(1/class_counts[class_counts!=0]))
            elif weight_type == 4:
                """
                version 4: 1/c(i)**0.5 (Inverse of Square Root of Number of Samples)
                """
                ret.append(1/class_counts[i]**0.5 / np.sum(1/class_counts[class_counts!=0]**0.5))
            elif weight_type == 5:
                """
                version 5: (1-beta) / (1-beta**c(i)) (Effective Number of Samples)
                https://medium.com/gumgum-tech/handling-class-imbalance-by-introducing-sample-weighting-in-the-loss-function-3bdebd8203b4
                """
                beta = 0.99999
                ret.append((1-beta)/(1-beta**class_counts[i])/ np.sum((1-beta)/(1-beta**class_counts[class_counts!=0])))
            else:
                ret.append(1/LABEL_NUM)
    return np.array(ret)

### Try to build models from fb SlowFast

In [20]:
import torch.utils.data as data
import numpy as np
import os
import sys
from PIL import Image
import torch
import random
import math
from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import argparse
from torch.autograd import Variable
from torchinfo import summary

from fvcore.nn import FlopCountAnalysis

print(torchvision.__version__)
print(torch.__version__)

data_root_dir = "/scratch1/zeyut/eat_detection/"

sys.path.append("../reimplementation/")
from models import fb_models
from models.helper import defaults, head_helper
import model_loader

0.9.1+cu102
1.9.1+cu102


In [2]:
model, model_type, inference_type = model_loader.get_models("x3d-s")

In [4]:
cur_device = torch.cuda.current_device()

In [5]:
cur_device

0

In [7]:
slowfast = fb_models.SlowFast('../reimplementation/models/config/SLOWFAST_4x16_R50.yaml')

In [3]:
input = torch.randn(4, 32, 3, 224, 224,dtype=torch.float)

In [11]:
output.shape

torch.Size([4, 3])

In [3]:
x3d = fb_models.X3D('../reimplementation/models/config/X3D_S.yaml')

192 432 2048 400 [13, 5, 5] 0.5 softmax True 1e-05 0.1


In [3]:
summary(x3d, input_size=(16, 16, 3, 312, 312))

Layer (type:depth-idx)                   Output Shape              Param #
├─VideoModelStem: 1-1                    [16, 24, 16, 156, 156]    --
|    └─X3DStem: 2-1                      [16, 24, 16, 156, 156]    --
|    |    └─Conv3d: 3-1                  [16, 24, 16, 156, 156]    648
|    |    └─Conv3d: 3-2                  [16, 24, 16, 156, 156]    120
|    |    └─BatchNorm3d: 3-3             [16, 24, 16, 156, 156]    48
|    |    └─ReLU: 3-4                    [16, 24, 16, 156, 156]    --
├─ResStage: 1-2                          [16, 24, 16, 78, 78]      --
|    └─ResBlock: 2-2                     [16, 24, 16, 78, 78]      --
|    |    └─X3DTransform: 3-5            [16, 24, 16, 78, 78]      5,240
|    |    └─Conv3d: 3-6                  [16, 24, 16, 78, 78]      576
|    |    └─BatchNorm3d: 3-7             [16, 24, 16, 78, 78]      48
|    |    └─ReLU: 3-8                    [16, 24, 16, 78, 78]      --
|    └─ResBlock: 2-3                     [16, 24, 16, 78, 78]      --
|    |   

In [9]:
output = model(input)

In [30]:
checkpoint = torch.load("../reimplementation/pre_trained/x3d_l.pyth")


In [31]:
x3d.load_state_dict(checkpoint['model_state'])

<All keys matched successfully>

In [15]:
ckpt_slowfast = torch.load("../reimplementation/pre_trained/SLOWFAST_4x16_R50.pkl",encoding='bytes')


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xca in position 0: invalid continuation byte

In [24]:
slowfast.load_state_dict(ckpt_slowfast['model_state'])

<All keys matched successfully>

### Train models

In [11]:
sys.path.append("../reimplementation/")
from train_model import val_loop
from torch.optim.lr_scheduler import ExponentialLR
from utils import AverageMeter,RateMeter

In [12]:
def class_weights(label_list):
    class_counts = []
    for label in range(LABEL_NUM):
        class_counts.append(np.sum(label_list==label))
    class_counts = np.array(class_counts)
    total = np.sum(class_counts)
    weights = []
    for i in range(LABEL_NUM):
        if class_counts[i] == 0:
            weights.append(0)
        else:
            weights.append(1/class_counts[i]**0.5)
            #weights.append(total/(class_counts[i]*LABEL_NUM))
    weights = np.array(weights) 
    weights = weights/np.sum(weights)
    return weights, class_counts

In [13]:
weights, _ = class_weights(train_set.label_list)
loss_fn = nn.CrossEntropyLoss(weight=torch.from_numpy(weights).float().cuda())


In [14]:
model = x3d_model
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
scheduler = ExponentialLR(optimizer, gamma=0.9)

In [33]:
def train_loop(dataloader, model, loss_fn, optimizer, maximum_step = 10000):
    model.train()
    train_loss = AverageMeter()
    train_acc = RateMeter() 
    for step, (input, target) in enumerate(dataloader):
        # Make predictions
        input = input.type(torch.cuda.FloatTensor)
        input = Variable(input).cuda()
        target = Variable(target).cuda()
        # Compute prediction and loss
        output = model(input)
        if output.dim() > 2:
            # output is frame-wise prediction
            # then change the dimension form [-1,T,C] to [-1,C,T], to fit in loss function
            output = output.permute(0, 2, 1)           
        # Backpropagation           
        loss = loss_fn(output, target)
        optimizer.zero_grad()
        loss.backward()
        cur_loss = loss.item()
        optimizer.step()            
        #update results
        pred = output.argmax(1)
        correct = (pred == target).sum().item()     
        if target.dim() == 1:
            total_num = target.size()[0]
        else:
            total_num = target.size()[0]*target.size()[1]
        cur_acc = correct/total_num
        train_loss.update(cur_loss)
        train_acc.update(correct,total_num) 
        tepoch.set_postfix(loss=f"{(cur_loss):>0.6f}", accuracy=f"{(100*cur_acc):>0.1f}%")            
        # Maximum train step per epoch is 10000
        if step >= maximum_step:
            break

#### Test validation loop

In [ ]:
model.eval()
val_loss = AverageMeter()
val_acc = RateMeter()    
val_tpr = [RateMeter() for _ in range(LABEL_NUM)]  
with torch.no_grad():
    for step, (input, target) in enumerate(val_loader):
        input = input.type(torch.cuda.FloatTensor)
        input = Variable(input).cuda()
        target = Variable(target).cuda()
        output = model(input)
        if output.dim() > 2:
            # output is frame-wise prediction
            # then change the dimension form [-1,T,C] to [-1,C,T], to fit in loss function
            output = output.permute(0, 2, 1)              
        #update results
        loss = loss_fn(output, target)
        print(loss)
        sys.stdout.flush() 
        val_loss.update(loss.item())
        if val_loss.avg > 10.00:
            print(loss)
            break

tensor(1.0983, device='cuda:0')
tensor(1.1044, device='cuda:0')
tensor(1.0973, device='cuda:0')
tensor(1.1040, device='cuda:0')
tensor(1.1039, device='cuda:0')
tensor(1.1010, device='cuda:0')
tensor(1.0961, device='cuda:0')
tensor(1.1047, device='cuda:0')
tensor(1.1009, device='cuda:0')
tensor(1.0989, device='cuda:0')
tensor(1.1055, device='cuda:0')
tensor(1.1032, device='cuda:0')
tensor(1.0944, device='cuda:0')
tensor(1.0984, device='cuda:0')
tensor(1.1025, device='cuda:0')
tensor(1.0969, device='cuda:0')
tensor(1.0994, device='cuda:0')
tensor(1.1055, device='cuda:0')
tensor(1.1038, device='cuda:0')
tensor(1.1001, device='cuda:0')
tensor(1.1017, device='cuda:0')
tensor(1.1000, device='cuda:0')
tensor(1.1043, device='cuda:0')
tensor(1.1001, device='cuda:0')
tensor(1.1031, device='cuda:0')
tensor(1.0984, device='cuda:0')
tensor(1.1009, device='cuda:0')
tensor(1.0967, device='cuda:0')
tensor(1.0986, device='cuda:0')
tensor(1.1046, device='cuda:0')
tensor(1.1042, device='cuda:0')
tensor(1

In [ ]:
for step, (input, target) in enumerate(val_loader):
    loss = loss_fn(output, target)
 

In [1]:
output.size()

NameError: name 'output' is not defined

In [ ]:
for epoch in range(1):
    sys.stdout.flush() 


Exception ignored in: <function tqdm.__del__ at 0x150e521220d0>
Traceback (most recent call last):
  File "/home/zeyut/.conda/envs/torch-1.8/lib/python3.8/site-packages/tqdm/std.py", line 1145, in __del__
    self.close()
  File "/home/zeyut/.conda/envs/torch-1.8/lib/python3.8/site-packages/tqdm/notebook.py", line 283, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm' object has no attribute 'disp'
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x150e51fa2160>
Traceback (most recent call last):
  File "/home/zeyut/.conda/envs/torch-1.8/lib/python3.8/site-packages/torch/utils/data/dataloader.py", line 1328, in __del__
    self._shutdown_workers()
  File "/home/zeyut/.conda/envs/torch-1.8/lib/python3.8/site-packages/torch/utils/data/dataloader.py", line 1320, in _shutdown_workers
Exception ignored in: Exception ignored in: <function tqdm.__del__ at 0x150e521220d0><function tqdm.__del__ at 0x150e521220d0>
Traceback (most recent

### Test if dataloader shuffle every time re-enumerating it

In [21]:
class testDataset(data.Dataset):
    def __init__(self):
        'Initialization'
        self.datalist = np.array([1,2,3,4,5,6,7,8,9,10])
        
    def __len__(self):
        return len(self.datalist)

    def __getitem__(self, idx):
        return self.datalist[idx]

In [22]:
testDataset = testDataset()
test_loader = data.DataLoader(
        dataset=testDataset,
        batch_size=3,
        shuffle=True,
    )

In [23]:
from tqdm import tqdm
max_step = 1
for epoch in range(5):
    with tqdm(test_loader,unit= "batch", total=max_step) as tepoch:
        for step, (data) in enumerate(tepoch):
            print(epoch, step,data)
            if step >= max_step:
                break


100%|██████████| 1/1 [00:00<00:00, 1035.63batch/s]

0 0 tensor([6, 4, 2])
0 1 tensor([9, 8, 3])
1 0 tensor([ 5, 10,  7])
1 1 tensor([6, 1, 3])
2 0 tensor([ 7, 10,  8])
2 1 tensor([9, 2, 5])
3 0 tensor([4, 1, 5])
3 1 tensor([6, 3, 7])
4 0 tensor([1, 7, 6])
4 1 tensor([3, 9, 2])


### Test model on test set

In [43]:
class testDataset(data.Dataset):
    def __init__(self,data_path,seq_len,stride,transform):
        'Initialization'
        self.data_path = data_path
        self.seq_len = seq_len
        self.stride = stride
        self.transform = transform
        self.input_list = []
        self.video_labels = []
        self.frame_names = []
        self._get_data_list(data_path,seq_len,stride)   
        
    def __len__(self):
        return len(self.input_list)
    
    def __getitem__(self, idx):
        frame_list = self.input_list[idx]
        frames = self._get_frames(frame_list)
        frames = torch.stack([transforms.functional.to_tensor(frame) for frame in frames])
        frames = self.transform(frames)
        return frames
    
    def _get_frames(self, frame_list):
        frames = []
        for frame_loc in frame_list:
            frames.append(Image.open(frame_loc).convert('RGB'))
        return frames
    
    def _get_data_list(self,data_path,seq_len,stride):
        frame_locs = []
        f = open(os.path.join(data_path,"gt_frame_3labels.txt"),"r")
        gt_frame = [str.split(line, "\t") for line in f.readlines()]
        sys.stdout.flush()
        for frame_info in gt_frame:
            self.frame_names.append(frame_info[0])
            frame_locs.append(os.path.join(data_path,frame_info[0]))
            cur_label_idx = LABEL_TABLE[str.split(frame_info[1], "\n")[0]]
            self.video_labels.append(cur_label_idx)
        for i in range(0, len(frame_locs)-seq_len+1, stride):
            self.input_list.append(frame_locs[i:i+seq_len])
        print(len(frame_locs),i)
        self.video_labels = np.array(self.video_labels)       
        self.frame_names = np.array(self.frame_names) 
            

class RateMeter(object):
    """Computes and stores the average rate (acc, TPR, etc)"""
    def __init__(self):
        self.reset()
    def reset(self):
        self.correctCount = 0
        self.totalCount = 0
        self.rate = 0
    def update(self, correct, total):
        self.correctCount += correct
        self.totalCount += total
        if self.totalCount:
            self.rate = self.correctCount / self.totalCount
        else:
            self.rate = 0
            

In [44]:
model_type = 1
test_batch_size=100
test_stride=1
test_video_list = [train_video_list[1]]
model.eval()
preprocess = transforms.Compose([
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                ])


print("{} videos in testing set".format(len(test_video_list)))
print("testing batch size: {}\n".format(test_batch_size))
acc = RateMeter()
total_nonintake = 0
for video_name in test_video_list:
    test_set = testDataset(
                data_path=os.path.join(FRAME_LOC,"train_set",video_name),
                seq_len=seq_len,
                stride=test_stride,
                transform=preprocess
                )
    test_loader = data.DataLoader(
                    dataset=test_set,
                    batch_size=batch_size,
                    shuffle=False,
                    num_workers=10,
                    pin_memory=True
                    )   
    video_labels = test_set.video_labels
    frame_names = test_set.frame_names

    pred_list = []
    prob_list = [] 
    '''
    model_type=1: seq2seq prediction
    get frame wise predictions using max vote strategy
    model_type=2: seq2one prediction
    model directly outputs final frame-wise prediction
    '''    
    with tqdm(test_loader,unit= "batch") as tepoch:
        tepoch.set_description(f"video {video_name}")
        for input in tepoch:
            input = Variable(input).cuda()
            output = model(input)
            cur_prob = output.detach().cpu()
            cur_prob = cur_prob.numpy()
        pred_list.append(cur_prob.argmax(-1))
        prob_list.append(cur_prob)  
    pred_list = np.concatenate(pred_list, axis=0)
    prob_list = np.concatenate(prob_list, axis=0)
    if model_type == 1:   
        # build a heat map
        heat_map = np.zeros((len(video_labels),LABEL_NUM))
        for seq_idx in range(len(pred_list)):
            for frame_idx in range(len(pred_list[seq_idx])):
                heat_map[test_stride*seq_idx+frame_idx][pred_list[seq_idx][frame_idx]] += 1
        video_preds = np.argmax(heat_map, axis=-1)
    elif model_type == 2:
        video_preds = np.zeros(len(video_labels))
        video_preds[seq_len-1:seq_len-1+len(pred_list)] = pred_list  

1 videos in testing set
testing batch size: 100



/home/zeyut/.conda/envs/torch-1.8/lib/python3.8/site-packages/torch/utils/data/dataloader.py:474: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
  0%|          | 0/299 [00:00<?, ?batch/s]

4771 4769


NameError: name 'tbatch' is not defined